# A3.4 · MCP is not a security boundary

**Function A — Security Architecture & Platform → The Platform & Cloud Security Engineer**  ·  *Security of AI*

---

**Risk.** Connector chaining as privilege escalation; unpinned servers.

**Control.** Server scanning, version pinning with hash verification, self-hosting.

**This lab.** Escalate through connector chaining, then pin and scan to stop it.

| | |
|---|---|
| Open-source tooling | kmcp, MCP Inspector, Sigstore |
| Open-weight models | GLM-4.6 |

> Runs anywhere: standard library only, no network, no API key. Where a lesson names a real tool you would deploy (Falco, OPA, SPIRE, Keycloak), the notebook models the *decision* that tool makes, so the lesson still lands on a machine that cannot pull containers.

In [ ]:
# --- Cyber Commons bootstrap -------------------------------------------------
# Puts the lab library on the path. Works from a clone, from the repo root, and
# on Kaggle. Standard library only — nothing to install, no network required.
import sys, os, subprocess
from pathlib import Path

def _find_labs():
    for base in [Path.cwd(), *Path.cwd().parents]:
        if (base / "labs" / "cybercommons" / "__init__.py").is_file():
            return base / "labs"
    # Kaggle kernels start in /kaggle/working with the repo absent. If the
    # kernel has internet enabled we clone it; if not, this raises and the
    # message tells you to attach the repo as a dataset instead.
    dest = Path("/kaggle/working/cyber-commons")
    if not dest.exists():
        subprocess.run(["git", "clone", "--depth", "1", "--branch", "claude/vulnbench-setup-scheduling-81aqov",
                        "https://github.com/spbreed/cyber-commons", str(dest)], check=True)
    return dest / "labs"

sys.path.insert(0, str(_find_labs()))
import cybercommons
print(cybercommons.banner("A3.4"))

MCP is a transport and a discovery mechanism. It is not a security boundary, and treating it as one is how tool servers end up with ambient authority.

In [ ]:
from cybercommons import injection, sandbox

# An MCP-style server exposes tools. The protocol says nothing about *who*
# may call them or what the content it returns is allowed to trigger.
SERVER_TOOLS = ["read_issue", "post_comment", "merge_pr"]

# A document fetched through one MCP tool contains an instruction for the agent.
poisoned = SERVER_TOOLS and injection.CORPUS[2].text
print("content returned by read_issue:\n ", poisoned, "\n")

naive  = injection.Deputy("agent", {"merge_pr"}, trust_data_as_instructions=True)
strict = injection.Deputy("agent", {"merge_pr"}, trust_data_as_instructions=False)

for name, d in (("MCP alone", naive), ("MCP + provenance", strict)):
    r = d.handle(poisoned, "merge_pr", source="mcp-tool-result")
    print(f"{name:20s} merge_pr executed={r['executed']}  blocked_by={r['blocked_by']}")

The protocol carried the payload faithfully. Whether it becomes an action depends entirely on a control that MCP does not specify — so you have to supply it.

In [ ]:
box = sandbox.ToolPolicy(allow={"read_issue"},
                         require_approval={"post_comment"},
                         deny={"merge_pr"})
for t in SERVER_TOOLS:
    print(box.check(t))

### Expect

With MCP alone the poisoned issue body triggers `merge_pr`. With provenance enforced it is blocked because the instruction came from a tool result rather than the principal. The tool policy independently denies `merge_pr` regardless.

### Your turn

List the MCP servers your developers have connected. For each, name what would happen if the content it returns were attacker-controlled. That list is your actual injection surface.

---

[All lessons](https://github.com/spbreed/cyber-commons/tree/claude/vulnbench-setup-scheduling-81aqov/labs/notebooks) · [Lesson page](https://spbreed.github.io/cyber-commons/lessons/A3.4.html) · [Lab library](https://github.com/spbreed/cyber-commons/tree/claude/vulnbench-setup-scheduling-81aqov/labs/cybercommons)

*Cyber Commons — a free, open commons for Cyber AI.*